In [85]:
# imports
import numpy as np
import ot  # Python Optimal Transport
import warnings
import sys

# Robust import to avoid name collisions / env errors
from rfphate import RFPHATE

from phate import PHATE # We need the base PHATE class for the final step
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox # For image plotting

# Sklearn imports
from sklearn.datasets import load_digits, fetch_openml
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Evaluation metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, silhouette_score



# Silence harmless warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)


# SPGW CLASS

In [86]:
import numpy as np
import ot  # Python Optimal Transport
from rfphate import RFPHATE # Import the function
from phate import PHATE     # Import the base class
from scipy.spatial.distance import pdist, squareform


class SupervisedPotentialGW(object):
    """
    Supervised Manifold Alignment using RF-Potential Distances
    and Gromov-Wasserstein.
    """
    
    def __init__(self, **kwargs):
        """
        Initialize the alignment model.
        
        Parameters
        ----------
        **kwargs :
            Other arguments passed to the rfphate.RFPHATE and
            phate.PHATE constructors (e.g., n_components, t,
            n_estimators, etc.).
        """
        # No lambda_inter, we use MALI-style fusion
        self.embedder_params = kwargs
        
        self.T = None 
        self.W_combined = None
        self.z_a = None
        self.z_b = None
        self.n_a = 0
        self.n_b = 0
        self.embedding_ = None

    def fit(self, x_a, y_a, x_b, y_b):
        """
        Fits the alignment model.
        """
        print("Fitting Domain A (Optical Digits)...")
        self.n_a = x_a.shape[0]
        rfphate_a = RFPHATE(y=y_a, **self.embedder_params)
        self.z_a = rfphate_a.fit_transform(x_a, y_a)
        
        print("Fitting Domain B (USPS Digits)...")
        self.n_b = x_b.shape[0]
        rfphate_b = RFPHATE(y=y_b, **self.embedder_params)
        self.z_b = rfphate_b.fit_transform(x_b, y_b)

        # Store the RF-PHATE operator for later use
        self.rfphate_op = rfphate_a

        print("Extracting RF-potential distances...")
        # Now these potentials are correctly supervised
        potential_a = rfphate_a.phate_op.diff_potential
        potential_b = rfphate_b.phate_op.diff_potential
        
        d_a = squareform(pdist(potential_a, metric='euclidean'))
        d_b = squareform(pdist(potential_b, metric='euclidean'))
        
        prox_a = rfphate_a.proximity.toarray()
        prox_b = rfphate_b.proximity.toarray()

        print("Computing Gromov-Wasserstein alignment...")
        # This T will now be based on supervised geometries
        p_a = ot.unif(self.n_a)
        p_b = ot.unif(self.n_b)
        
        self.T = ot.gromov_wasserstein(
            d_a, d_b, p_a, p_b, 'square_loss', verbose=False, max_iter=100
        )
        
        print("Building joint affinity matrix (MALI-style fusion)...")
        
        # 1. Normalize T
        T_max = self.T.max()
        T_norm = self.T / T_max if T_max > 0 else self.T
        
        # 2. Transport affinities: W_ab = (W_a * T_norm) + (T_norm * W_b)
        W_ab = prox_a.dot(T_norm) + T_norm.dot(prox_b)
        W_ba = prox_b.dot(T_norm.T) + T_norm.T.dot(prox_a)

        # 3. Symmetrize the final block
        W_ab = (W_ab + W_ba.T) / 2
        W_ba = W_ab.T

        # 4. Build the final combined affinity matrix W
        self.W_combined = np.block([
            [prox_a, W_ab],
            [W_ba,   prox_b]
        ])
        
        print("Model fit complete.")
        return self

    def fit_transform(self, x_a=None, y_a=None, x_b=None, y_b=None):
        """
        Embeds the joint kernel using PHATE.
        """
        if x_a is not None and y_a is not None and \
        x_b is not None and y_b is not None:
            self.fit(x_a, y_a, x_b, y_b)

        if self.W_combined is None:
            raise RuntimeError("You must call fit() before fit_transform().")

        print("Computing joint PHATE embedding from W_combined...")

        self.embedding_ = self.rfphate_op.phate_op.fit_transform(self.W_combined)

        print("Joint embedding complete.")
        return self.embedding_
            

# Fused GW

In [87]:
import numpy as np
import ot  # Python Optimal Transport
from rfphate import RFPHATE # Import the function
from phate import PHATE     # Import the base class
from scipy.spatial.distance import pdist, squareform


class FusedGromovWasserstein(object):
    """
    Supervised Manifold Alignment using RF-Potential Distances
    and Fused Gromov-Wasserstein (FGW).
    
    This version uses FGW to incorporate label information directly
    into the alignment step, preventing class mixing.
    
    UPDATED: This version uses the diff_potential matrix *directly*
    as the geometry, instead of incorrectly processing it with pdist.
    """
    
    def __init__(self, alpha=0.5, **kwargs):
        """
        Initialize the alignment model.
        
        Parameters
        ----------
        alpha : float, optional (default=0.5)
            The trade-off parameter for Fused Gromov-Wasserstein.
            - alpha=0: Pure Wasserstein (label matching)
            - alpha=1: Pure Gromov-Wasserstein (geometry matching)
            A value of 0.5 balances both.
            
        **kwargs :
            Other arguments passed to the rfphate.RFPHATE and
            phate.PHATE constructors (e.g., n_components, t,
            n_estimators, etc.).
        """
        self.embedder_params = kwargs
        self.alpha = alpha  # Store the FGW trade-off parameter
        
        self.T = None 
        self.W_combined = None
        self.z_a = None
        self.z_b = None
        self.n_a = 0
        self.n_b = 0
        self.embedding_ = None
        self.rfphate_op = None # Store one of the operators

    def fit(self, x_a, y_a, x_b, y_b):
        """
        Fits the alignment model.
        """
        print("Fitting Domain A (Optical Digits)...")
        self.n_a = x_a.shape[0]
        # Ensure labels are 1D arrays
        y_a = y_a.ravel() 
        rfphate_a = RFPHATE(y=y_a, **self.embedder_params)
        self.z_a = rfphate_a.fit_transform(x_a, y_a)
        
        print("Fitting Domain B (USPS Digits)...")
        self.n_b = x_b.shape[0]
        # Ensure labels are 1D arrays
        y_b = y_b.ravel()
        rfphate_b = RFPHATE(y=y_b, **self.embedder_params)
        self.z_b = rfphate_b.fit_transform(x_b, y_b)

        # Store the RF-PHATE operator for later use in fit_transform
        self.rfphate_op = rfphate_a

        print("Extracting RF-potential (diffused) geometries...")
        # --- KEY CHANGE ---
        # The potential matrix *is* the geometry.
        # We assume it is (n_a, n_a) and (n_b, n_b).
        # We NO LONGER pass this to pdist.
        d_a = rfphate_a.phate_op.diff_potential
        d_b = rfphate_b.phate_op.diff_potential
        

        # --- END KEY CHANGE ---
        
        # We still need the original proximities for the MALI-style fusion
        prox_a = rfphate_a.proximity.toarray()
        prox_b = rfphate_b.proximity.toarray()

        # Build Label Distance Matrix M
        print("Creating label distance matrix M...")
        Y_a = y_a.reshape(-1, 1) # Shape (n_a, 1)
        Y_b = y_b.reshape(1, -1) # Shape (1, n_b)
        M = (Y_a != Y_b).astype(float)
        
        print("Computing Fused Gromov-Wasserstein alignment...")
        p_a = ot.unif(self.n_a)
        p_b = ot.unif(self.n_b)
        
        # Call ot.fused_gromov_wasserstein
        # We are now aligning the *actual* diffused geometries (d_a, d_b)
        # guided by the label matrix M.
        self.T = ot.fused_gromov_wasserstein(
            M, d_a, d_b, p_a, p_b, 
            alpha=self.alpha,  # Use the trade-off parameter
            loss_fun='square_loss', 
            verbose=False, 
            max_iter=100
        )
        
        print("Building joint affinity matrix (MALI-style fusion)...")
        
        # 1. Normalize T
        T_max = self.T.max()
        T_norm = self.T / T_max if T_max > 0 else self.T
        
        # 2. Transport affinities: W_ab = (W_a * T_norm) + (T_norm * W_b)
        #    We use the *non-diffused* proximities for the fusion step,
        #    which seems consistent with MALI-style logic.
        W_ab = prox_a.dot(T_norm) + T_norm.dot(prox_b)
        W_ba = prox_b.dot(T_norm.T) + T_norm.T.dot(prox_a)

        # 3. Symmetrize the final block
        W_ab = (W_ab + W_ba.T) / 2
        W_ba = W_ab.T

        # 4. Build the final combined affinity matrix W
        self.W_combined = np.block([
            [prox_a, W_ab],
            [W_ba,   prox_b]
        ])
        
        print("Model fit complete.")
        return self

    def fit_transform(self, x_a=None, y_a=None, x_b=None, y_b=None):
        """
        Embeds the joint kernel using PHATE.
        """
        if x_a is not None and y_a is not None and \
           x_b is not None and y_b is not None:
            self.fit(x_a, y_a, x_b, y_b)

        if self.W_combined is None:
            raise RuntimeError("You must call fit() before fit_transform().")
            
        if self.rfphate_op is None:
            raise RuntimeError("RFPHATE operator not fitted. Call fit() first.")

        print("Computing joint PHATE embedding from W_combined...")

        # Use the stored PHATE operator to embed the joint affinity matrix
        # We use the base phate_op, not the full rfphate_op
        self.embedding_ = self.rfphate_op.phate_op.fit_transform(self.W_combined)

        print("Joint embedding complete.")
        return self.embedding_

# RF-MALI extension

In [ ]:
import numpy as np
import ot  # Python Optimal Transport
from rfphate import RFPHATE # Import the function
from phate import PHATE     # Import the base class
from scipy.spatial.distance import pdist, squareform, cdist
from sklearn.preprocessing import normalize

class RFMALI(object):
    """
    RF-MALI: Semi-Supervised Manifold Alignment using RFGAP label bridge
    """

    def __init__(self, **kwargs):
        """
        Initialize the alignment model.
        
        Parameters
        ------------
        **kwargs :
            Other arguments passed to the rfphate.RFPHATE and
            phate.PHATE constructors (e.g., n_components, t,
            n_estimators, etc.).
        """
        self.embedder_params = kwargs        
        self.T = None  # Coupling matrix
        self.W_combined = None  # Combined affinity matrix
        self.z_a = None
        self.z_b = None
        self.n_a = 0
        self.n_b = 0
        self.embedding_ = None
        self.rfphate_op = None # Store one of the operators

    def _get_rfgap_posteriors(self, rfphate_obj, y, common_labels):
        pass

    def fit(self, x_a, y_a, x_b, y_b):
        """
        Fits the alignment model.
        """
        print("Fitting Domain A (Optical Digits)...")
        self.n_a = x_a.shape[0]
        y_a = y_a.ravel() 
        rfphate_a = RFPHATE(y=y_a, **self.embedder_params)
        # self.z_a = rfphate_a.fit_transform(x_a, y_a)
        
        print("Fitting Domain B (USPS Digits)...")
        self.n_b = x_b.shape[0]
        y_b = y_b.ravel()
        rfphate_b = RFPHATE(y=y_b, **self.embedder_params)
        # self.z_b = rfphate_b.fit_transform(x_b, y_b)

        # Store the RF-PHATE operator
        self.rfphate_op = rfphate_a

        # We do not diffuse, unlike MALI-style, because RFGAP already encodes adaptative supervised geometry
        # # Get *both* sets of matrices
        # print("Extracting RF-potential (diffused) matrices...")
        # d_a = rfphate_a.phate_op.diff_potential
        # d_b = rfphate_b.phate_op.diff_potential
        
        print("Extracting RF proximities (for final fusion)...")
        prox_a = rfphate_a.proximity.toarray()
        prox_b = rfphate_b.proximity.toarray()

        print("Building RFGAP posteriors...")
        labels_a_set = np.unique(y_a)
        labels_b_set = np.unique(y_b)
        common_labels = np.intersect1d(labels_a_set, labels_b_set)
        
        if len(common_labels) < 1:
            raise ValueError("No common labels found.")
        

        
        
        print("Optimal Transport...")
        # uniform weights of value 1 for domain A
        p_a = np.ones(self.n_a, dtype=float)
        # uniform weights of value n_a/n_b for domain B (straightforward, see MALI paper)
        p_b = np.ones(self.n_b, dtype=float) * (float(self.n_a) / float(self.n_b))

        print(f"p_a: shape={p_a.shape}, first5={p_a[:5]}")
        print(f"p_b: shape={p_b.shape}, first5={p_b[:5]}")

        # Call ot.fused_gromov_wasserstein
        # M = MALI-style cosine cost (W part)
        # d_a, d_b = Global potential geometry (GW part)
        self.T = ot.wasserstein(
            M, d_a, d_b, p_a, p_b, 
            loss_fun='square_loss', 
            verbose=False,
            max_iter=100
        )
        
        print("Building joint affinity matrix (MALI-style fusion)...")
        
        T_max = self.T.max()
        T_norm = self.T / T_max if T_max > 0 else self.T
        
        # Use the *original proximities* for the final fusion
        W_ab = prox_a.dot(T_norm) + T_norm.dot(prox_b)
        W_ba = prox_b.dot(T_norm.T) + T_norm.T.dot(prox_a)

        W_ab = (W_ab + W_ba.T) / 2
        W_ba = W_ab.T

        # Use the *original proximities* on the diagonal
        self.W_combined = np.block([
            [prox_a, (1-lam) * W_ab],
            [(1-lam) * W_ba,   prox_b]
        ])
        
        print("Model fit complete.")
        return self

    def fit_transform(self, x_a=None, y_a=None, x_b=None, y_b=None):
        """
        Embeds the joint kernel using PHATE.
        """
        if x_a is not None and y_a is not None and \
           x_b is not None and y_b is not None:
            self.fit(x_a, y_a, x_b, y_b)

        if self.W_combined is None:
            raise RuntimeError("You must call fit() before fit_transform().")
            
        if self.rfphate_op is None:
            raise RuntimeError("RFPHATE operator not fitted. Call fit() first.")

        print("Computing joint PHATE embedding from W_combined...")

        self.embedding_ = self.rfphate_op.phate_op.fit_transform(self.W_combined)

        print("Joint embedding complete.")
        return self.embedding_

In [97]:
# datasets (1797 + 9298) is not feasible for a demo.
# We'll take a stratified subsample of 500 points from each.
N_SAMPLES = 500

def stratified_subsample(X, y, n_samples):
    """Helper function to subsample data while preserving label ratios."""
    if n_samples >= len(y):
        return X, y
    
    # Use StratifiedKFold to get indices
    skf = StratifiedKFold(n_splits=int(len(y) / n_samples))
    # We just use the indices from the first fold
    train_idx, test_idx = next(skf.split(X, y))
    
    # Select the smaller split (test_idx)
    if len(test_idx) < n_samples:
        # This is a bit of a hack, but we'll take the larger split
        # if the test split is too small.
        if len(train_idx) < n_samples:
             # Failsafe if data is tiny
            return X, y
        indices = train_idx[:n_samples]
    else:
        indices = test_idx[:n_samples]
        
    return X[indices], y[indices]

# Domain A: Optical Digits (8x8)
print("Loading Optical Digits (Domain A)...")
digits = load_digits()
x_a, y_a = digits.data, digits.target
print(f"Original opti-digits shape: {x_a.shape}")

# Domain B: USPS Digits (16x16)
print("Loading USPS Digits (Domain B)...")
# This will download the data if not cached
usps = fetch_openml('USPS', version=1, as_frame=False)
x_b, y_b = usps.data, usps.target.astype(int) # Labels are strings
print(f"Original USPS shape: {x_b.shape}")

# Preprocessing: Scale data
x_a = StandardScaler().fit_transform(x_a)
x_b = StandardScaler().fit_transform(x_b)

# Subsample the data
print(f"Subsampling to {N_SAMPLES} points per domain...")
x_a_s, y_a_s = stratified_subsample(x_a, y_a, N_SAMPLES)
x_b_s, y_b_s = stratified_subsample(x_b, y_b, N_SAMPLES)

print(f"Domain A shape (final): {x_a_s.shape}")
print(f"Domain B shape (final): {x_b_s.shape}")



Loading Optical Digits (Domain A)...
Original opti-digits shape: (1797, 64)
Loading USPS Digits (Domain B)...
Original USPS shape: (9298, 256)
Subsampling to 500 points per domain...
Domain A shape (final): (500, 64)
Domain B shape (final): (500, 256)


# Data loading and preprocessing

In [98]:
# We MUST subsample. GW is O(N^3), so running on the full
# datasets (1797 + 9298) is not feasible for a demo.
# We'll take a stratified subsample of 500 points from each.
N_SAMPLES = 500

def stratified_subsample(X, y, n_samples):
    """Helper function to subsample data while preserving label ratios."""
    if n_samples >= len(y):
        return X, y
    
    # Use StratifiedKFold to get indices
    skf = StratifiedKFold(n_splits=int(len(y) / n_samples))
    # We just use the indices from the first fold
    train_idx, test_idx = next(skf.split(X, y))
    
    # Select the smaller split (test_idx)
    if len(test_idx) < n_samples:
        # This is a bit of a hack, but we'll take the larger split
        # if the test split is too small.
        if len(train_idx) < n_samples:
             # Failsafe if data is tiny
            return X, y
        indices = train_idx[:n_samples]
    else:
        indices = test_idx[:n_samples]
        
    return X[indices], y[indices]

# Domain A: Optical Digits (8x8)
print("Loading Optical Digits (Domain A)...")
digits = load_digits()
x_a, y_a = digits.data, digits.target
print(f"Original opti-digits shape: {x_a.shape}")

# Domain B: USPS Digits (16x16)
print("Loading USPS Digits (Domain B)...")
# This will download the data if not cached
usps = fetch_openml('USPS', version=1, as_frame=False)
x_b, y_b = usps.data, usps.target.astype(int) # Labels are strings
print(f"Original USPS shape: {x_b.shape}")

# Preprocessing: Scale data
x_a = StandardScaler().fit_transform(x_a)
x_b = StandardScaler().fit_transform(x_b)

# Subsample the data
print(f"Subsampling to {N_SAMPLES} points per domain...")
x_a_s, y_a_s = stratified_subsample(x_a, y_a, N_SAMPLES)
x_b_s, y_b_s = stratified_subsample(x_b, y_b, N_SAMPLES)

print(f"Domain A shape (final): {x_a_s.shape}")
print(f"Domain B shape (final): {x_b_s.shape}")



Loading Optical Digits (Domain A)...
Original opti-digits shape: (1797, 64)
Loading USPS Digits (Domain B)...
Original USPS shape: (9298, 256)
Subsampling to 500 points per domain...
Domain A shape (final): (500, 64)
Domain B shape (final): (500, 256)


# Model training and alignment

In [99]:
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, silhouette_score

# ---
# 5. MODEL TRAINING & ALIGNMENT
# ---
# (Using your specified parameters)
# Initialize the model
model = RFMALI(
    n_components=2,      # Final embedding dim
    t='auto',            # PHATE t parameter (denoising)
    n_estimators=200,    # RF trees
    random_state=42,
    n_jobs=-1,
    verbose=0
)

# Fit and transform
print("\nStarting alignment...")
%time embedding = model.fit_transform(x_a_s, y_a_s, x_b_s, y_b_s)
print("Alignment complete.")

# ---
# 6. VISUALIZATION (Color-blind friendly)
# ---
print("\n--- Visualization ---")
print("Generating plots (color-blind friendly)...")

# Create helper arrays for plotting
labels_combined = np.concatenate([y_a_s, y_b_s])
domains_combined = np.concatenate([
    np.full(N_SAMPLES, "Optidigits (A)"),
    np.full(N_SAMPLES, "USPS (B)")
])

plt.figure(figsize=(16, 7))

# --- Plot 1: Colored by Domain (with Color-blind safe colors + Markers) ---
plt.subplot(1, 2, 1)

# Using 'blue' and 'darkorange' - a common color-blind safe pair
domain_viz = {
    "Optidigits (A)": {'color': 'blue', 'marker': 'o'}, # Circles for Domain A
    "USPS (B)": {'color': 'darkorange', 'marker': 'X'}  # X's for Domain B
}

for domain in np.unique(domains_combined):
    idx = (domains_combined == domain)
    plt.scatter(
        embedding[idx, 0], embedding[idx, 1],
        c=domain_viz[domain]['color'],
        marker=domain_viz[domain]['marker'],
        label=domain,
        alpha=0.6, s=25
    )
plt.title("Aligned Embedding (by Domain)", fontweight="bold")
plt.legend()
plt.xlabel("PHATE 1")
plt.ylabel("PHATE 2")

# --- Plot 2: Colored by Label (with 'Paired' cmap + Markers) ---
plt.subplot(1, 2, 2)

# Use 'Paired' colormap - designed for categorical data
# Also add distinct markers for each digit.
cmap_10_safe = plt.get_cmap("Paired")
markers_10 = ['o', 's', '^', 'v', 'P', '*', 'D', 'X', 'p', 'H'] # 10 distinct markers

for label in range(10):
    idx = (labels_combined == label)
    plt.scatter(
        embedding[idx, 0], embedding[idx, 1],
        color=cmap_10_safe(label / 10.), # Normalize label for colormap
        marker=markers_10[label],  # Use distinct markers
        label=str(label),
        alpha=0.8, s=30
    )
plt.title("Aligned Embedding (by Label)", fontweight="bold")
plt.legend(title="Digit", markerscale=1.5, bbox_to_anchor=(1.05, 1))
plt.xlabel("PHATE 1")
plt.ylabel("PHATE 2")

plt.suptitle("SupervisedPotentialGW Alignment: Optical Digits vs. USPS", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# ---
# 7. QUANTITATIVE EVALUATION
# ---
print("\n--- Quantitative Evaluation ---")

# 1. Label Preservation (k-NN Accuracy)
print("1. Label Preservation (k-NN Accuracy):")
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(embedding, labels_combined)
y_pred = knn.predict(embedding)
knn_acc = accuracy_score(labels_combined, y_pred)
print(f"  k-NN Accuracy on embedding: {knn_acc:.4f} (Ideal: High)")

# 2. Label Preservation (Silhouette Score)
print("\n2. Label Preservation (Silhouette Score):")
label_sil = silhouette_score(embedding, labels_combined)
print(f"  Silhouette Score (by Label): {label_sil:.4f} (Ideal: High, near 1.0)")

# 3. Domain Mixing (Silhouette Score)
print("\n3. Domain Mixing (Silhouette Score):")
# domains_combined already created above for plotting
domain_sil = silhouette_score(embedding, domains_combined)
print(f"  Silhouette Score (by Domain): {domain_sil:.4f} (Ideal: Low, near 0.0)")

print("\n--- Interpretation ---")
print(f"A good alignment shows:")
print(f"  - High k-NN Acc & high Label Silhouette (labels are well-separated)")
print(f"  - Low Domain Silhouette (domains are well-mixed)")


Starting alignment...
Fitting Domain A (Optical Digits)...
Fitting Domain B (USPS Digits)...
Extracting RF proximities (for final fusion)...
Building RFGAP posteriors...
Optimal Transport...
p_a: [0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002
 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002 0.002


SystemExit: 1